# PySpark — SQL in PySpark

PySpark lets you run **standard SQL queries** on DataFrames by registering them as temporary views.  
This is useful for people familiar with SQL, for complex queries, and for migrating existing SQL pipelines.

```python
df.createOrReplaceTempView("table_name")
spark.sql("SELECT * FROM table_name WHERE salary > 80000")
```

| Approach | Use when |
|----------|----------|
| DataFrame API (`filter`, `groupBy`, etc.) | Code-first, programmatic, type-safe |
| `spark.sql()` | SQL-first, complex queries, existing SQL teams |

Both produce identical execution plans — no performance difference.

## Setup

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("SQL-in-PySpark") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

employees = spark.createDataFrame([
    (1, "Alice",   "Engineering", "NY", 95000, 5),
    (2, "Bob",     "Marketing",   "CA", 72000, 3),
    (3, "Charlie", "Engineering", "NY", 110000, 8),
    (4, "Diana",   "HR",          "TX", 65000, 2),
    (5, "Eve",     "Marketing",   "CA", 85000, 6),
    (6, "Frank",   "Engineering", "TX", 98000, 7),
], ["emp_id", "name", "dept", "state", "salary", "years"])

departments = spark.createDataFrame([
    ("Engineering", "New York",   500000),
    ("Marketing",   "California", 300000),
    ("HR",          "Texas",      150000),
], ["dept", "hq_city", "budget"])

print("DataFrames created.")

## 1. createOrReplaceTempView() — Register as SQL Table

Registers the DataFrame as a **session-scoped temporary view** — like a table you can query with SQL.  
`createOrReplace` means if the view already exists it is replaced, not errored.

In [ ]:
# Register views
employees.createOrReplaceTempView("employees")
departments.createOrReplaceTempView("departments")

# List all registered views
spark.catalog.listTables()

## 2. spark.sql() — Run SQL Queries

The result of `spark.sql()` is a regular DataFrame — all DataFrame operations apply to it.

In [ ]:
# Basic SELECT
spark.sql("SELECT * FROM employees").show()

# WHERE clause
spark.sql("""
    SELECT name, dept, salary
    FROM employees
    WHERE salary > 80000
    ORDER BY salary DESC
""").show()

# The result is a DataFrame — chain more operations
result = spark.sql("SELECT * FROM employees WHERE dept = 'Engineering'")
print("Type:", type(result))   # pyspark.sql.dataframe.DataFrame
result.count()

## 3. Aggregations in SQL

In [ ]:
# GROUP BY with HAVING — much more natural in SQL
spark.sql("""
    SELECT
        dept,
        COUNT(*)          AS headcount,
        SUM(salary)       AS total_salary,
        ROUND(AVG(salary),0) AS avg_salary,
        MAX(salary)       AS top_salary
    FROM employees
    GROUP BY dept
    HAVING COUNT(*) >= 2
    ORDER BY total_salary DESC
""").show()

## 4. JOINs in SQL

In [ ]:
# JOIN employees with departments
spark.sql("""
    SELECT
        e.name,
        e.dept,
        e.salary,
        d.hq_city,
        d.budget
    FROM employees e
    JOIN departments d ON e.dept = d.dept
    ORDER BY e.salary DESC
""").show()

# LEFT JOIN — keep all employees even if no dept match
spark.sql("""
    SELECT e.name, e.dept, d.hq_city
    FROM employees e
    LEFT JOIN departments d ON e.dept = d.dept
""").show()

## 5. Subqueries and CTEs

`WITH` (Common Table Expressions) make complex SQL readable — PySpark supports them fully.

In [ ]:
# Subquery — employees earning above department average
spark.sql("""
    SELECT name, dept, salary
    FROM employees e
    WHERE salary > (
        SELECT AVG(salary)
        FROM employees
        WHERE dept = e.dept
    )
    ORDER BY dept, salary DESC
""").show()

# CTE — cleaner version
spark.sql("""
    WITH dept_avg AS (
        SELECT dept, AVG(salary) AS avg_sal
        FROM employees
        GROUP BY dept
    )
    SELECT e.name, e.dept, e.salary, ROUND(d.avg_sal, 0) AS dept_avg
    FROM employees e
    JOIN dept_avg d ON e.dept = d.dept
    WHERE e.salary > d.avg_sal
    ORDER BY e.dept
""").show()

## 6. Window Functions in SQL

Window functions (`RANK`, `ROW_NUMBER`, `LAG`, `LEAD`) work the same in PySpark SQL as standard SQL.

In [ ]:
# Rank employees by salary within their department
spark.sql("""
    SELECT
        name, dept, salary,
        RANK()       OVER (PARTITION BY dept ORDER BY salary DESC) AS rank,
        ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
        ROUND(AVG(salary) OVER (PARTITION BY dept), 0)            AS dept_avg
    FROM employees
    ORDER BY dept, rank
""").show()

## 7. SQL vs DataFrame API — Side by Side

In [ ]:
from pyspark.sql.functions import col, avg, count, sum as spark_sum, round as spark_round

# --- SQL approach ---
sql_result = spark.sql("""
    SELECT dept, COUNT(*) AS headcount, ROUND(AVG(salary), 0) AS avg_salary
    FROM employees
    WHERE years >= 5
    GROUP BY dept
    ORDER BY avg_salary DESC
""")

# --- DataFrame API approach (identical result) ---
df_result = (
    employees
    .filter(col("years") >= 5)
    .groupBy("dept")
    .agg(
        count("*").alias("headcount"),
        spark_round(avg("salary"), 0).alias("avg_salary")
    )
    .orderBy(col("avg_salary").desc())
)

print("SQL result:")
sql_result.show()
print("DataFrame API result:")
df_result.show()

## 8. createGlobalTempView — Shared Across Sessions

`createOrReplaceTempView` is session-scoped. `createGlobalTempView` survives across SparkSessions and lives in `global_temp` schema.

In [ ]:
# Global view — must prefix table name with global_temp.
employees.createOrReplaceGlobalTempView("employees_global")

# Access with global_temp. prefix
spark.sql("SELECT COUNT(*) FROM global_temp.employees_global").show()

print("Temp view: session-scoped, disappears when SparkSession ends")
print("Global view: shared across sessions, use global_temp.table_name to query")

## Quick Reference

```python
# Register a view
df.createOrReplaceTempView("my_table")
df.createOrReplaceGlobalTempView("my_global")  # global_temp.my_global

# Run SQL
spark.sql("SELECT * FROM my_table").show()
spark.sql("""
    SELECT dept, COUNT(*) AS n, AVG(salary) AS avg
    FROM my_table
    GROUP BY dept
    HAVING COUNT(*) > 1
    ORDER BY avg DESC
""").show()

# CTE
spark.sql("""
    WITH ranked AS (
        SELECT *, RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rk
        FROM my_table
    )
    SELECT * FROM ranked WHERE rk = 1
""")

# List registered tables
spark.catalog.listTables()
```

## Interview Questions

1. What is `createOrReplaceTempView()`? How long does the view last?
2. What is the difference between `createOrReplaceTempView` and `createOrReplaceGlobalTempView`?
3. Is there a performance difference between `spark.sql()` and the DataFrame API?
4. How do you pass a Python variable into a `spark.sql()` query safely?
5. Can you chain DataFrame operations after `spark.sql()`? Why?
6. How do you list all registered temporary views?
7. What SQL features does PySpark support that the DataFrame API makes harder?

## Practice Exercises

**Easy:**
1. Register `employees` as a temp view and query all employees from Engineering using SQL.
2. Use SQL to count employees per department, ordered by count descending.

**Medium:**
3. Write a SQL query that finds employees earning above the company-wide average salary.
4. Use a CTE to find the top earner in each department.

**Advanced:**
5. Rewrite one of your SQL queries using the DataFrame API — verify both return identical results.
6. Write a SQL window function query: for each employee, show their salary rank within their department and their salary vs department average.

In [ ]:
spark.stop()